# Tutorial Denoising

> denoising


In [ ]:
#| default_exp tutorial_multispectral

In [ ]:
from bioMONAI.data import *
from bioMONAI.transforms import *
from bioMONAI.core import *
from bioMONAI.core import Path
from bioMONAI.data import get_image_files, get_target, RandomSplitter
from bioMONAI.nets import BasicUNet, DynUNet
from bioMONAI.losses import *
from bioMONAI.losses import SSIMLoss
from bioMONAI.metrics import *
from bioMONAI.datasets import download_file, split_dataframe


In [ ]:
import warnings
warnings.filterwarnings("ignore")


In [ ]:
device = get_device()
print(device)

cuda


In [ ]:
# Specify the directory where you want to save the downloaded files
data_folder = "../_data/rxrx1_subset_monai"
# Define the base URL for the dataset
url = "https://github.com/Project-MONAI/MONAI-extra-test-data/releases/download/0.8.1/rxrx1_subset_monai.zip"

download_file(url, data_folder, extract=True, hash='e80db433db641bb390ade991b81f98814a26c7de30e0da6f20e8abddf7a84538', extract_dir='')

The file has been downloaded and saved to: ../_data/rxrx1_subset_monai
Extracted files have been saved to: ../_data/rxrx1_subset_monai


In [ ]:
data_folder += '/rxrx1_subset_monai/'
csv_file = data_folder+'metadata.csv'
split_dataframe(csv_file, train_fraction=0.7, valid_fraction=0.1, split_column='dataset', add_is_valid=True, train_path="train.csv", test_path="test.csv", valid_path="valid.csv", data_save_path=data_folder)

Train and test files saved as '../_data/rxrx1_subset_monai/rxrx1_subset_monai/train.csv' and '../_data/rxrx1_subset_monai/rxrx1_subset_monai/test.csv' respectively.
'is_valid' column added to '../_data/rxrx1_subset_monai/rxrx1_subset_monai/train.csv' for validation samples.


In [ ]:
X_path = data_folder+'images'

bs = 16
patch_size = 128

itemTfms = [RandCropND(patch_size), RandRot90(prob=.75), RandFlip(prob=0.75)]
batchTfms = [ScaleIntensityPercentiles()]

data = BioDataLoaders.class_from_csv(
    data_folder,
    'train.csv',
    fn_col=2,
    label_col=4,
    valid_col=-1,
    seed=42, 
    img_class=BioImage,
    item_tfms=itemTfms,
    batch_tfms=batchTfms, 
    show_summary=False,
    bs = bs,
    )

# print length of training and validation datasets
print('train images:', len(data.train_ds.items), '\nvalidation images:', len(data.valid_ds.items))

TypeError: expected str, bytes or os.PathLike object, not DataFrame